<a href="https://colab.research.google.com/github/ivazquez805/cpe470-embeddedML/blob/main/Project_1_(spring_2025)_BeanDiseaseClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bean Disease Classifier Project
For this assignment you'll take what you've learned so far and build a classifier for bean disease. You'll be provided with training and validation data based on 224x224 pixel color images taken of bean plants in Uganda. These images show healthy bean leaves as well as 2 types of common disease: bean rust and angular leaf spots. Your job will be to build a neural network that can tell the difference between the healthy and diseased leaves.

Your assignment is to:
1. Build the colab as-is, making sure everything works
2. Improve the model by:
  
  a. Changing the [parameters of the image generator](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image/ImageDataGenerator)
  
  b. Changing the width / [depth of the neural network](https://www.tensorflow.org/guide/keras/sequential_model)

  c. Chaning the [loss function](https://www.tensorflow.org/api_docs/python/tf/keras/losses) / [optimizer](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers)

  d. changing the number of EPOCHS

  Be sure to read the APIs and explore. This will help you with your final project.

  Your report will be a listing of the different classifiers your tried (the above combinations), the accuracy and the model training times. From this summary, you should also make observations about the data. Why did the changes improve (or not) the classifier?

  Since there are three people per team, I'd suggest you split up the work. Also, be smart! If you tried every combination, your runtime will get pretty long.

In [ ]:
# Do not change this code
!pip install --upgrade --no-cache-dir gdown

In [ ]:
# Do not change this code
!gdown "https://storage.googleapis.com/learning-datasets/beans/train.zip" -O /tmp/train.zip
!gdown "https://storage.googleapis.com/learning-datasets/beans/validation.zip" -O /tmp/validation.zip
!gdown "https://storage.googleapis.com/learning-datasets/beans/test.zip" -O /tmp/test.zip

In [ ]:
# Do not change this code
import os
import zipfile

local_zip = '/tmp/train.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/tmp')
local_zip = '/tmp/validation.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/tmp')
local_zip = '/tmp/test.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/tmp/test')
zip_ref.close()

Now you need to define a generator to process the data we have loaded in Colab so that our model can use it for training. As we showed in the previous video you'll first have to define an ```ImageDataGenerator``` and then flow the data into it.

*A hint: You don't want abnormal data!*

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
  rescale=1./255
)

validation_datagen = ImageDataGenerator(
  rescale=1./255,
)

TRAIN_DIRECTORY_LOCATION = '/tmp/train'
VAL_DIRECTORY_LOCATION = '/tmp/validation'
TARGET_SIZE = (224,224)
CLASS_MODE = 'categorical'

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIRECTORY_LOCATION,
    target_size = TARGET_SIZE,
    batch_size = 128,
    class_mode = CLASS_MODE
)

validation_generator = validation_datagen.flow_from_directory(
    VAL_DIRECTORY_LOCATION,
    target_size = TARGET_SIZE,
    batch_size = 128,
    class_mode = CLASS_MODE,
    shuffle = False
)

Now its your turn to define a model to learn this data.

*A hint: Like with the CIFAR-10 assignment, your model may want to learn some high level features and then classify them. This time it may help to make the model a little wider at times.*

In [ ]:
import tensorflow as tf
model = tf.keras.models.Sequential([
   # Find the features with Convolutions and Pooling
   tf.keras.layers.Conv2D(16, (3,3), activation='relu', input_shape=(224, 224, 3)),
   tf.keras.layers.MaxPooling2D(4, 4),
   tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
   tf.keras.layers.MaxPooling2D(4,4),
   # Flatten the results to feed into a DNN
   tf.keras.layers.Flatten(),
   # 512 neuron hidden layer
   tf.keras.layers.Dense(512, activation='relu'),
   tf.keras.layers.Dense(3, activation='softmax')
])

# This will print a summary of your model when you're done!
model.summary()

Then you'll need to pick an appropriate loss function and optimizer.

*A hint: remember we are classifying again.*

In [ ]:
LOSS_FUNCTION = 'categorical_crossentropy'
OPTIMIZER = 'adam'

model.compile(
    loss = LOSS_FUNCTION,
    optimizer = OPTIMIZER,
    metrics = ['accuracy']
)

Finally select the number of epochs you'd like to train for and train your model!

*A hint: something in the low tens is a good place to start*

In [ ]:
NUM_EPOCHS = 5

history = model.fit(
      train_generator,
      epochs = NUM_EPOCHS,
      verbose = 1,
      validation_data = validation_generator)

# summarize history for accuracy
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.xlim([0,NUM_EPOCHS])
plt.ylim([0.4,1.0])
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score

# Get predictions on validation set
y_pred_probs = model.predict(validation_generator)

# Convert predictions to class indices
y_pred = np.argmax(y_pred_probs, axis=1)

# True labels (already stored in generator)
y_true = validation_generator.classes

# Compute balanced accuracy
bal_acc = balanced_accuracy_score(y_true, y_pred)

print("Balanced Accuracy:", bal_acc)
